<a href="https://colab.research.google.com/github/normala127/NLP_Yelp_Review_Project/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data Analytics Project
## Predicting Restaurant Failure: An NLP Driven Risk Assesment from Yelp Reviews

Students: Hatidza Imamovic, Asja Basovic

### 1. Dataset creation and environment setup

Three datasets are needed to create the final dataset which will be used for analysis and model training. These are:
- business.json: holds data about each restaurant
- review.json: holds all the reviews for each restaurant
- checkin.json: holds the dates of all checked in visits in a given restaurant

Each is loaded and then combined in regards to the business_id to ensure a correct join. The final output is saved as final.json.


In [3]:
!pip install pyspark
!apt-get install openjdk-11-jdk-headless -qq > /dev/null


In [14]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [15]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("project").getOrCreate()

print(spark)

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
def show_shape(df):
  print((df.count(), len(df.columns)))

Loading the first dataset: business.json

In [18]:
df_business = spark.read.option("mode", "PERMISSIVE").json(r"/content/drive/MyDrive/yelp_academic_dataset_business.json")
df_business.printSchema()
df_business.show()

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    |-- GoodForDancing: str

Loading the second dataset: review.json

In [19]:
df_review = spark.read.json(r"/content/drive/MyDrive/yelp_academic_dataset_review.json")

In [ ]:
show_shape(df_business)

In [ ]:
show_shape(df_review)

Joining df_review and df_business into joined_df

In [20]:
df_review.createOrReplaceTempView("review")
df_business.createOrReplaceTempView("business")

In [21]:
joined_df = spark.sql("""
SELECT t1.*, t2.*
FROM review t1
LEFT JOIN business t2 ON t2.business_id = t1.business_id
""")
joined_df.show(5)

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+-------+-------------+-----------+--------------------+-----------+------------+-----+-----+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|             address|          attributes|         business_id|          categories|        city|               hours|is_open|     latitude|  longitude|                name|postal_code|review_count|stars|state|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+-------+-------------+-----------+--------------------+-----------+------

In [22]:
joined_df = joined_df.drop(df_business['business_id'])

In [ ]:
show_shape(joined_df)

Loading the third dataset: checkin.json

In [23]:
df_checkin = spark.read.option("mode", "PERMISSIVE").json(r"/content/drive/MyDrive/yelp_academic_dataset_checkin.json")
df_checkin.printSchema()
df_checkin.show()

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)

+--------------------+--------------------+
|         business_id|                date|
+--------------------+--------------------+
|---kPU91CF4Lq2-Wl...|2020-03-13 21:10:...|
|--0iUa4sNDFiZFrAd...|2010-09-13 21:43:...|
|--30_8IhuyMHbSOcN...|2013-06-14 23:29:...|
|--7PUidqRWpRSpXeb...|2011-02-15 17:12:...|
|--7jw19RH9JKXgFoh...|2014-04-21 20:42:...|
|--8IbOsAAxjKRoYsB...|2015-06-06 01:03:...|
|--9osgUCSDUWUkoTL...|2015-06-13 02:00:...|
|--ARBQr1WMsTWiwOK...|2014-12-12 00:44:...|
|--FWWsIwxRwuw9vIM...|2010-09-11 16:28:...|
|--FcbSxK1AoEtEAxO...|2017-08-18 19:43:...|
|--LC8cIrALInl2vyo...|2017-01-12 19:10:...|
|--MbOh2O1pATkXa7x...|2013-04-21 01:52:...|
|--N9yp3ZWqQIm7DqK...|2012-10-06 20:46:...|
|--O3ip9NpXTKD4oBS...|2010-04-17 21:07:...|
|--OS_I7dnABrXvRCC...| 2018-05-11 18:23:36|
|--S43ruInmIsGrnnk...|2010-08-29 01:17:...|
|--SJXpAa0E-GCp2sm...|2014-04-06 22:23:...|
|--Sd93OFWITqDHifM...|2013-01-09 17

In [24]:
df_checkin_new=df_checkin.withColumnRenamed('date', 'date_checkin')
df_checkin_new.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date_checkin: string (nullable = true)



Joining joined_df (review+business) and df_checkin into joined_df2

In [25]:
df_checkin_new.createOrReplaceTempView('checkin')
joined_df.createOrReplaceTempView('joined_df')

In [26]:
joined_df2 = spark.sql("""
SELECT t1.*, t2.date_checkin
FROM joined_df t1
JOIN checkin t2 ON t2.business_id = t1.business_id
""")
joined_df2.show(5)

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|

In [27]:
joined_df2.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: 

Checking the distribution of opened and closed retaurants

In [ ]:
df_isOpen=joined_df2.filter(joined_df2['is_open']==1)
df_isOpen.show()

In [ ]:
df_isClosed=joined_df2.filter(joined_df2['is_open']==0)
df_isClosed.show()

In [ ]:
show_shape(df_isOpen) # 80% is open


In [ ]:
show_shape(df_isClosed) # 20% is closed

Dropping a part of open restaurants based on business_id to balance out the classes

In [28]:
from pyspark.sql.functions import col, hash, count

In [29]:
# id-level labels
id_labels = joined_df2.select("business_id", "is_open").distinct()

majority_ids = id_labels.filter(col("is_open") == 1) \
    .withColumn("keep", (hash("business_id") % 10000) < 5)  # keep 10%

In [30]:
minority_ids = id_labels.filter(col("is_open") == 0) \
    .withColumn("keep", (hash("business_id") % 100) < 50)  # keep 50%

In [31]:
ids_to_keep = majority_ids.filter("keep").union(
    minority_ids.select("business_id", 'is_open', 'keep')
)

balanced_df = joined_df2.join(ids_to_keep.select("business_id"),
                      "business_id")

In [ ]:
show_shape(balanced_df)

In [ ]:
#balanced_df.show()

In [ ]:
show_shape(majority_ids)

In [ ]:
show_shape(minority_ids)

In [32]:
balanced_df.groupBy('is_open').agg(count('review_id').alias('c')).show()

+-------+-------+
|is_open|      c|
+-------+-------+
|      0|1180155|
|      1|2831060|
+-------+-------+



In [33]:
stratified_df = balanced_df.sampleBy('is_open', fractions={0: 0.25, 1: 0.10})

In [34]:
stratified_df.groupBy('is_open').agg(count('review_id').alias('c')).show()

+-------+------+
|is_open|     c|
+-------+------+
|      0|589951|
|      1|595439|
+-------+------+



Renaming columns and dropping uneeded columns

In [35]:
stratified_df.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: 

In [36]:
cols=stratified_df.columns
stars_columns=[i for i, c in enumerate(cols) if c=='stars']

cols[stars_columns[0]]='stars_review'
cols[stars_columns[1]]='stars_business'

stratified_df=stratified_df.toDF(*cols)

In [37]:
stratified_df.columns

['business_id',
 'cool',
 'date',
 'funny',
 'review_id',
 'stars_review',
 'text',
 'useful',
 'user_id',
 'address',
 'attributes',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'stars_business',
 'state',
 'date_checkin']

In [38]:
final_df=stratified_df.drop(*['cool', 'funny', 'useful', 'latitude', 'longitude', 'postal_code', 'attributes', 'hours'])

In [39]:
final_df.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars_review: double (nullable = true)
 |-- text: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- city: string (nullable = true)
 |-- is_open: long (nullable = true)
 |-- name: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- stars_business: double (nullable = true)
 |-- state: string (nullable = true)
 |-- date_checkin: string (nullable = true)



In [40]:
show_shape(final_df)

(1183893, 15)


### 2. Preprocessing

Firstly, on a global level, null and duplicate values were dropped.

This section covers:
- lowercase,
- keep only letters (from all languages) and spaces,
- remove extra spaces,
- remove private information,
- emojis, urls.

It creates a pipeline for TF-IDF and for semantic analysis as slightly different cleaning techniques are used for each.

The section also uses n-grams, and handles "not" negation effectively.


In [41]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

from pyspark.sql.functions import when, col, count, sum
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType

from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, NGram, VectorAssembler
from pyspark.ml.functions import vector_to_array

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [42]:
df = final_df.select("*")

In [43]:
df.show()

+--------------------+-------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+------------+-------+--------------------+------------+--------------+-----+--------------------+
|         business_id|               date|           review_id|stars_review|                text|             user_id|             address|          categories|        city|is_open|                name|review_count|stars_business|state|        date_checkin|
+--------------------+-------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+------------+-------+--------------------+------------+--------------+-----+--------------------+
|-0eUa8TsXFFy0FCxH...|2016-09-10 14:47:36|V3sJTTreGBvJGmTm8...|         5.0|So glad I found w...|0KCOEsM1WGKYUg6ey...|      3131 Walnut St|Caterers, Sandwic...|Philadelphia|      0|Waterfront Gourme...|          26|           

In [44]:
null_counts = df.select([count(when(col(c).isNull(), c).alias(c)) for c in df.columns])

null_counts.show()

+--------------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------------------------+-----------------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------------------+--------------------------------------------------------------+-----------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------------------------------+-----------------------------------------------------------------------------------+--------------------------------------------------------+-----------------------------------------------------------------------------+
|c

In [45]:
df_nulls = df.filter(df['categories'].isNull())
df_nulls.limit(10)

DataFrame[business_id: string, date: string, review_id: string, stars_review: double, text: string, user_id: string, address: string, categories: string, city: string, is_open: bigint, name: string, review_count: bigint, stars_business: double, state: string, date_checkin: string]

In [ ]:
# TODO visaulaization

In [46]:
df = df.fillna({'categories': 'Unknown'})

In [47]:
df = df.dropDuplicates(['text'])
show_shape(df)

(1182990, 15)


In [48]:
def clean_sentiment(df):
  return df.withColumn("text_cleaned",
          F.trim(
            F.regexp_replace(
              F.lower(F.col("text")),
              r"http\S+|www\S+|\S+@\S+", " "
            ),
          )
      ).withColumn("text_cleaned_sentiment", F.regexp_replace(F.col("text_cleaned"), r"\s+", " "))


In [49]:
def clean_tfidf(df):

    return df.withColumn("text_cleaned",
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.lower(F.col("text")),
                    r"http\S+|www\S+|\S+@\S+", " "
                ),
                r'[^\p{L}\s]+', " "
            )
        )
    ).withColumn("text_cleaned", F.regexp_replace(F.col("text_cleaned"), r"\s+", " "))

In [51]:
# TODO SAMPLE
sdf = df.sampleBy('is_open', fractions={0: 0.001, 1: 0.001})
show_shape(sdf)

(11990, 15)


In [52]:
sdf=clean_tfidf(sdf)
#df.show()

In [53]:
sdf=clean_sentiment(sdf)

In [ ]:
#todo partitioning

In [ ]:
#show_shape(train_df)

In [ ]:
#show_shape(test_df)

Cleaning other columns (not including any IDs, review and business stars, review count and is open)

In [ ]:
from pyspark.sql.functions import split_part

def clean_columns(df):
  return sdf.withColumn('categories_cleaned',
          F.trim(F.lower(F.regexp_replace(F.col('categories'), r"[^a-z\s]", "")))
          ) \
          .withColumn("city_cleaned", F.trim(F.lower(F.col("city")))
          ) \
          .withColumn("state_cleaned", F.trim(F.lower(F.col("state")))
          ) \
          .withColumn("address_cleaned", F.trim(F.regexp_replace(F.col("address"), r"\s+", " "))
          ) \
          .withColumn("name_cleaned", F.trim(F.regexp_replace(F.col("name"), r"[^\p{L}\s]", ""))
          ) \
          .withColumn("date_parsed", F.to_date(F.col("date"), "yyyy-MM-dd HH:mm:ss")
          ) \
          .withColumnn("last_checkin_date", split_part(df.date_checkin, ",", -1)) #todo: add back to original dataset after testing

In [ ]:
df2 = sdf.select('categories', 'city', 'state', 'address', 'name', 'date', 'date_checkin')

In [ ]:
df2 = clean_columns(df2)
df2.show(5)

### 3. Feature engineering

In [ ]:
from pyspark.sql.functions import month, day, year

def extract_date():
  return sdf.withColumn('month', F.month('date_parsed')) \
  .withColumn('month_checkin', F.month('last_checkin_date')) \
  .withColumn('day', F.day('date_parsed')) \
  .withColumn('day_checkin', F.day('last_checkin_date')) \
  .withColumn('year', F.year('date_parsed')) \
  .withColumn('year_checkin', F.year('late_checkin_date'))


In [ ]:
import numpy as np

def encode_dates():
  return sdf.withColumn("month_sin", F.sin(2 * np.pi * F.col("month") / 12)) \
  .withColumn("month_cos", F.cos(2 * np.pi * F.col("month") / 12)) \
  .withColumn('day_sin', F.sin(2 * np.pi * F.col("day") / 7)) \
  .withColumn('day_cos', F.cos(2 * np.pi * F.col("day") / 7)) \
  .withColumn("month_checkin_sin", F.sin(2 * np.pi * F.col("month_checkin") / 12)) \
  .withColumn("month_checkin_cos", F.cos(2 * np.pi * F.col("month_checkin") / 12)) \
  .withColumn('day_checkin_sin', F.sin(2 * np.pi * F.col("day_checkin") / 7)) \
  .withColumn('day_checkin_cos', F.cos(2 * np.pi * F.col("day_checkin") / 7))


In [54]:
remover = StopWordsRemover()
default_stops = remover.getStopWords()
updated_stops = [w for w in default_stops if w != 'not']

def tfidf(vocab_size =10000, min_df = 500, suffix = ""):

  tokenizer = RegexTokenizer(
      inputCol="text_cleaned"+suffix,
      outputCol="tokens_raw"+suffix,
      pattern=r"\W+",
      gaps=True,
      toLowercase=True,
      minTokenLength=1
  )

  remover = StopWordsRemover(inputCol="tokens_raw"+suffix, outputCol="filtered_tokens"+suffix, stopWords=updated_stops)

  ngram = NGram(n=2, inputCol="filtered_tokens"+suffix, outputCol="bigrams"+suffix)

  uni_vectorizer = CountVectorizer(
        inputCol="filtered_tokens"+suffix,
        outputCol="uni_count_features"+suffix,
        vocabSize=vocab_size,
        minDF=min_df
    )

  bi_vectorizer = CountVectorizer(
      inputCol = 'bigrams'+suffix,
      outputCol = "bi_count_features"+suffix,
      vocabSize = vocab_size,
      minDF=min_df
  )

  assembler = VectorAssembler(
        inputCols=["uni_count_features"+suffix, "bi_count_features"+suffix],
        outputCol="combined_counts"+suffix
    )

  idf = IDF(inputCol="combined_counts"+suffix, outputCol="features"+suffix, minDocFreq=min_df)

  return  [tokenizer, remover, ngram, uni_vectorizer, bi_vectorizer, assembler, idf]


In [55]:
tfidf_pipeline_stages = tfidf()
tfidf_pipeline = Pipeline(stages=tfidf_pipeline_stages)

In [56]:
sdf = sdf.withColumn('stars_review_binary', when(df['stars_review']<3, 0).otherwise(1))

In [57]:
from pyspark.ml.classification import LogisticRegression, NaiveBayes, RandomForestClassifier

def get_log_reg(suffix):
    return LogisticRegression(
        featuresCol=f'features{suffix}',
        labelCol='stars_review_binary',
        predictionCol=f'prediction{suffix}',
        probabilityCol=f'probability{suffix}',
        regParam=0.3,
        maxIter=10
    )

def get_naive_bayes(suffix):
    return NaiveBayes(
        featuresCol=f'features{suffix}',
        labelCol='stars_review_binary',
        predictionCol=f'prediction{suffix}',
        probabilityCol=f'probability{suffix}',
        modelType="multinomial"
    )

def get_random_forest(suffix):
    return RandomForestClassifier(
        featuresCol=f'features{suffix}',
        labelCol='stars_review_binary',
        predictionCol=f'prediction{suffix}',
        numTrees=20
    )

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

models = [
    ("Logistic Regression", get_log_reg("_sentiment")),
    ("Naive Bayes", get_naive_bayes("_sentiment")),
    ("Random Forest", get_random_forest("_sentiment"))
]

results = []
evaluator = MulticlassClassificationEvaluator( #understand this
    labelCol="stars_review_binary",
    predictionCol="prediction_sentiment",
    metricName="accuracy"
)

for name, model in models:
    current_stages = tfidf(suffix="_sentiment")
    current_stages.append(model)

    pipeline = Pipeline(stages=current_stages)

    train_data, test_data = sdf.randomSplit([0.8, 0.2], seed=42)

    fitted_model = pipeline.fit(train_data)
    predictions = fitted_model.transform(test_data)

    accuracy = evaluator.evaluate(predictions)
    results.append((name, accuracy))
    print(f"{name} Accuracy: {accuracy:.4f}")

print("\Final Results")
for name, acc in results:
    print(f"{name}: {acc}")

In [ ]:
def sentiment_analysis():
  stages = tfidf(suffix="_sentiment")

  log_reg = LogisticRegression(featuresCol='features_sentiment',
                             labelCol='stars_review_binary',  predictionCol='prediction_sentiment', probabilityCol = 'probability_sentiment', regParam=0.3, maxIter=10)

  stages.append(log_reg)

  return stages

In [ ]:
sentiment_analysis_pipeline_stages = sentiment_analysis()
sentiment_analysis_pipeline = Pipeline(stages=sentiment_analysis_pipeline_stages)

In [ ]:
unique_bus_df = df.select("business_id", "is_open").distinct()

fractions = {0: 0.8, 1: 0.8} # 80% of closed (0) and 80% of open (1)

train_ids = unique_bus_df.sampleBy("is_open", fractions, seed=42)

test_ids = unique_bus_df.join(train_ids, on="business_id", how="left_anti")

train_df = df.join(train_ids.select("business_id"), on="business_id", how="inner")
test_df = df.join(test_ids.select("business_id"), on="business_id", how="inner")

sentiment_model = sentiment_analysis_pipeline.fit(train_df)
tfidf_model = tfidf_pipeline.fit(train_df)

df_with_sentiment = sentiment_model.transform(df)

df_clean = df_with_sentiment.withColumn("sentiment_score", vector_to_array(F.col("probability_sentiment"))[1]) \
                            .drop("tokens_raw_sentiment", "filtered_tokens_sentiment", "uni_count_features_sentiment", "bi_count_features_sentiment", "combined_counts_sentiment","features_sentiment", "probability_sentiment")

final_feature_df = tfidf_model.transform(df_clean)

In [ ]:
sentiment_model = sentiment_analysis_pipeline.fit(train_df)
tfidf_model = tfidf_pipeline.fit(train_df)

def prepare_features(input_df):
    df_with_sent = sentiment_model.transform(input_df)

    df_clean = df_with_sent.withColumn(
        "sentiment_score",
        vector_to_array(F.col("probability_sentiment"))[1]
    ).drop(
        "tokens_raw_sentiment", "filtered_tokens_sentiment",
        "uni_count_features_sentiment", "bi_count_features_sentiment",
        "combined_counts_sentiment", "features_sentiment", "probability_sentiment"
    )

    return tfidf_model.transform(df_clean)

train_final = prepare_features(train_df)
test_final = prepare_features(test_df)

Clean other columns
EDA
FE
Build and hypertune models (check literature)
Metrics and eval